<a href="https://colab.research.google.com/github/Sai-Kumar-Jonnalagadda/Reinforcement-Learning/blob/main/RL_ASSIGN_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
Modified Epsilon-Greedy Strategy for a Multi-Armed Bandit
-----------------------------------------------------------
This version is DETERMINISTIC: it uses the exact same random
numbers (r), rewards, and tie/exploration choices as the worked
solution, so running it reproduces that solution step by step.
"""

# ----------------------------------------------------------------
# PART A - Machine Initialization
# ----------------------------------------------------------------
machines = ['A', 'B', 'C']

# ----------------------------------------------------------------
# PART B - Knowledge base: N, R, Q tables (all start at zero)
# ----------------------------------------------------------------
N = {m: 0 for m in machines}      # number of times each machine was played
R = {m: 0 for m in machines}      # cumulative reward per machine
Q = {m: 0.0 for m in machines}    # current estimated value per machine

EPSILON = 0.2
N_PLAY = 10

# Predefined random numbers r, one per play (matches the worked solution)
r_values = [0.55, 0.63, 0.12, 0.80, 0.09, 0.71, 0.34, 0.18, 0.05, 0.11]

# Predefined rewards, one per play, in the order the plays occur
rewards = [3, 4, 6, 7, 4, 5, 8, 2, 3, 5]

# Predefined outcomes for every random "pick" needed:
#   - Play 1: tie-break among all-zero Q values      -> A
#   - Play 3: explore, excluded A, choose from {B,C}  -> B
#   - Play 5: explore, excluded B, choose from {A,C}  -> C
#   - Play 8: explore, excluded B, choose from {A,C}  -> A
#   - Play 9: explore, excluded B, choose from {A,C}  -> C
#   - Play 10: explore, excluded B, choose from {A,C} -> A
manual_picks = ['A', 'B', 'C', 'A', 'C', 'A']
pick_index = 0


def get_max_q_machine(q_table):
    """Return the machine name with the highest Q value.
    Ties are broken using the next value from manual_picks."""
    global pick_index
    max_q = max(q_table.values())
    best = [m for m in q_table if q_table[m] == max_q]
    if len(best) == 1:
        return best[0]
    choice = manual_picks[pick_index]
    pick_index += 1
    return choice


def choose_machine(q_table, epsilon, r):
    """
    PART C - Modified Epsilon-Greedy Decision
    ------------------------------------------
    r < epsilon  -> EXPLORE: choose from machines EXCLUDING the
                    current best (highest-Q) machine.
    r >= epsilon -> EXPLOIT: choose the current best (highest-Q) machine.
    """
    global pick_index
    best_machine = get_max_q_machine(q_table)

    if r < epsilon:
        candidates = [m for m in q_table if m != best_machine]
        if len(candidates) == 1:
            choice = candidates[0]
        else:
            choice = manual_picks[pick_index]
            pick_index += 1
        mode = "EXPLORE"
    else:
        choice = best_machine
        mode = "EXPLOIT"

    return choice, mode


def main():
    total_reward = 0

    for play in range(1, N_PLAY + 1):
        r = r_values[play - 1]
        reward = rewards[play - 1]

        machine, mode = choose_machine(Q, EPSILON, r)

        old_q = Q[machine]

        # ----------------------------------------------------------------
        # PART D - Reward Collection  (predefined instead of input() here)
        # ----------------------------------------------------------------
        # (In the interactive version this line would be:
        #  reward = float(input("Enter reward: ")) )

        # ----------------------------------------------------------------
        # PART E - Update N, R, Q  (incremental sample-average formula)
        # ----------------------------------------------------------------
        N[machine] += 1
        R[machine] += reward
        Q[machine] = Q[machine] + (reward - Q[machine]) / N[machine]

        total_reward += reward

        print(f"Play {play}: r={r:.2f} ({mode}) -> Machine {machine}, "
              f"Reward={reward}, Q({machine}): {old_q:.2f} -> {Q[machine]:.2f}, "
              f"N({machine})={N[machine]}")

    # ----------------------------------------------------------------
    # PART F - Display Results
    # ----------------------------------------------------------------
    print("\n" + "=" * 50)
    print("FINAL RESULTS AFTER 10 PLAYS")
    print("=" * 50)
    print(f"{'Machine':<10}{'N':<6}{'R':<8}{'Q':<8}")
    for m in machines:
        print(f"{m:<10}{N[m]:<6}{R[m]:<8}{Q[m]:<8.2f}")

    best = max(Q, key=Q.get)
    print(f"\nBest-performing machine (highest Q): {best}")
    print(f"Total reward collected over {N_PLAY} plays: {total_reward}")


if __name__ == "__main__":
    main()

Play 1: r=0.55 (EXPLOIT) -> Machine A, Reward=3, Q(A): 0.00 -> 3.00, N(A)=1
Play 2: r=0.63 (EXPLOIT) -> Machine A, Reward=4, Q(A): 3.00 -> 3.50, N(A)=2
Play 3: r=0.12 (EXPLORE) -> Machine B, Reward=6, Q(B): 0.00 -> 6.00, N(B)=1
Play 4: r=0.80 (EXPLOIT) -> Machine B, Reward=7, Q(B): 6.00 -> 6.50, N(B)=2
Play 5: r=0.09 (EXPLORE) -> Machine C, Reward=4, Q(C): 0.00 -> 4.00, N(C)=1
Play 6: r=0.71 (EXPLOIT) -> Machine B, Reward=5, Q(B): 6.50 -> 6.00, N(B)=3
Play 7: r=0.34 (EXPLOIT) -> Machine B, Reward=8, Q(B): 6.00 -> 6.50, N(B)=4
Play 8: r=0.18 (EXPLORE) -> Machine A, Reward=2, Q(A): 3.50 -> 3.00, N(A)=3
Play 9: r=0.05 (EXPLORE) -> Machine C, Reward=3, Q(C): 4.00 -> 3.50, N(C)=2
Play 10: r=0.11 (EXPLORE) -> Machine A, Reward=5, Q(A): 3.00 -> 3.50, N(A)=4

FINAL RESULTS AFTER 10 PLAYS
Machine   N     R       Q       
A         4     14      3.50    
B         4     26      6.50    
C         2     7       3.50    

Best-performing machine (highest Q): B
Total reward collected over 10 plays: